In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

RAW_DIR = PROJECT_ROOT / "data" / "raw_transcripts"
NORMALIZED_DIR = PROJECT_ROOT / "data" / "processed" / "normalized_transcripts"

NORMALIZED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw transcripts:", RAW_DIR)
print("Normalized output:", NORMALIZED_DIR)

Project root: c:\Users\malak\OneDrive\Desktop\NLP Project\NLP-project
Raw transcripts: c:\Users\malak\OneDrive\Desktop\NLP Project\NLP-project\data\raw_transcripts
Normalized output: c:\Users\malak\OneDrive\Desktop\NLP Project\NLP-project\data\processed\normalized_transcripts


In [2]:
from nomralization import normalize_arabic_for_rag

transcript_files = list(RAW_DIR.glob("*.txt"))

print("Found transcripts:", len(transcript_files))
for file in transcript_files:
    print("-", file.name)

Found transcripts: 4
- الأخطبوط  الدحيح.txt
- الساموراي  الدحيح.txt
- تاج محل  الدحيح.txt
- فيزياء و فلسفة الحركة  الدحيح.txt


Transcripts normalization

In [3]:
for file_path in transcript_files:
    raw_text = file_path.read_text(encoding="utf-8")
    normalized_text = normalize_arabic_for_rag(raw_text)

    output_path = NORMALIZED_DIR / file_path.name
    output_path.write_text(normalized_text, encoding="utf-8")

    print(f"Saved normalized transcript: {output_path.name}")

Saved normalized transcript: الأخطبوط  الدحيح.txt
Saved normalized transcript: الساموراي  الدحيح.txt
Saved normalized transcript: تاج محل  الدحيح.txt
Saved normalized transcript: فيزياء و فلسفة الحركة  الدحيح.txt


Sample Nomralized

In [4]:
sample_file = transcript_files[0]
raw_sample = sample_file.read_text(encoding="utf-8")[:1000]
normalized_sample = normalize_arabic_for_rag(raw_sample)

print("RAW SAMPLE:\n")
print(raw_sample)

print("\n" + "="*80 + "\n")

print("NORMALIZED SAMPLE:\n")
print(normalized_sample)

RAW SAMPLE:

6.45: يا ربي!
7.756: أنا زهقت!
8.396: ما تثبت يا ابني بقى،
9.564: اثبت، تعبتني!
10.846: سيب الجهاز يقرا!
11.712: بصراحة، أنا مش عاجبني موضوع التجربة دي،
13.843: أنا قُلت: الموضوع فيه Seafood،
15.775: فقُلت: هيبقى فيه أكلة سمك حلوة.
17.875: يا حبيبي، التجربة دي لو نجحت،
19.521: هتبقى أول إنسان
نُص بني آدم، ونُص أخطبوط في التاريخ.
23.864: وإيه المبهر لمّا أبقى نُص أخطبوط؟!
25.41: دا عادي، يتّاكل جنب السُبّيط
وشوية رُز صيادية،
28.616: عادي، مش حاجة فظيعة يعني.
30.037: صيادية إيه؟! انت بتقول إيه؟!
32.233: انت هيبقى عندك قدرات الأخطبوط،
هيبقى عندك قدرات خارقة،
35.285: هتقدر مثلًا تعمل Multi-Tasking.
37.296: إيه يعني Multi-Tasking؟!
38.715: أي موظف، ولّا أي طفل في مدرسة،
41.481: بيعمل موضوع الـMulti-Tasking دا.
43.111: طب إيه رأيك، أنا الصبح محاسب،
44.737: الضهر Designer،
45.928: بالليل "أوبر"،
46.882: وبأعمل "تراكات" راب.
48.037: يا سيدي، بلاش،
50.185: كفاية إن الأخطبوط عنده 3 قلوب،
52.781: متخيل؟!
53.733: في دي، عندك حق، أنا عندي قلب واحد،
56.002: بس القلب الواحد دا، بعون الله

Chunking

In [11]:
from chunking import build_chunks_from_folder

chunks = build_chunks_from_folder(
    normalized_dir=NORMALIZED_DIR,
    chunk_size=350,
    overlap=70
)

print("Total chunks:", len(chunks))
print("Example chunk keys:", chunks[0].keys())
print("Example episode:", chunks[0]["episode"])
print("Example chunk ID:", chunks[0]["chunk_id"])

ImportError: cannot import name 'build_chunks_from_folder' from 'chunking' (c:\Users\malak\OneDrive\Desktop\NLP Project\NLP-project\src\chunking.py)

Chunking Sample

In [ ]:
print(chunks[0]["text"][:1500])

Save chunks as JSON

In [ ]:
import json

CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "chunks.json"

with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print("Saved chunks to:", CHUNKS_PATH)

Embedding Step

In [ ]:
from embeddings import EmbeddingModel

embedding_model = EmbeddingModel()

chunk_texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = embedding_model.embed_texts(chunk_texts)

print("Number of chunks:", len(chunk_texts))
print("Embedding shape:", chunk_embeddings.shape)

Test one query embedding

In [ ]:
query = "الأخطبوط ذكي ازاي؟"

query_embedding = embedding_model.embed_query(query)

print("Query embedding shape:", query_embedding.shape)